## processed_data.csvを作成

### step0:生データを1行に（SMILES数値変換済）

In [1]:
import pandas as pd

def create_master_dataset():
    print("--- 📂 全特徴量を結合したマスターデータの作成開始 ---")
    
    # 1. 各ステップの中間データを絶対パスで読み込む
    path_clean = r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\01_cleaned_raw.csv"
    path_desc = r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\02_molecular_descriptors.csv"
    
    df_clean = pd.read_csv(path_clean)
    df_desc = pd.read_csv(path_desc)
    
    # 2. インデックスをリセットしてズレを防止（念のため）
    df_clean = df_clean.reset_index(drop=True)
    df_desc = df_desc.reset_index(drop=True)
    
    # 3. df_clean（基本情報 + 環境 + WVP）と df_desc（分子記述子）を横方向に結合
    # ※ ご提示いただいた列順（基本情報 → 分子記述子 → 環境・WVP）になるように並び替えます
    
    # 基本情報の列
    base_cols = ['id', 'experiment_id', 'composition_id', 'material_id', 'name', 'structure_id', 'material_concentration', 'smiles', 'proportion']
    # 環境・ターゲットの列
    env_cols = ['humidity', 'temperature', 'wvp']
    
    # 結合処理
    df_master = pd.concat([df_clean[base_cols], df_desc, df_clean[env_cols]], axis=1)
    
    # 4. 指定された絶対パスに保存
    export_path = r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\merged_basic_descriptors.csv"
    df_master.to_csv(export_path, index=False, encoding='utf-8-sig')
    
    print("\n" + "="*40)
    print(" 🎉 マスターデータセットの作成が完了しました！")
    print("="*40)
    print(f"・データ件数 (行数) : {df_master.shape[0]} 行")
    print(f"・総特徴量数 (列数) : {df_master.shape[1]} 列")
    print(f"・保存先パス: \n  {export_path}")
    print("="*40)
    
    return df_master

if __name__ == "__main__":
    try:
        df_master = create_master_dataset()
    except FileNotFoundError as e:
        print(f"⚠️ ファイルが見つかりません。Step 1 と Step 2 が完了しているか確認してください。\nエラー内容: {e}")

--- 📂 全特徴量を結合したマスターデータの作成開始 ---

 🎉 マスターデータセットの作成が完了しました！
・データ件数 (行数) : 5572 行
・総特徴量数 (列数) : 33 列
・保存先パス: 
  C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\merged_basic_descriptors.csv


### step1:各行ごとにmolを計算


In [2]:
import pandas as pd

df = pd.read_csv(r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\merged_basic_descriptors.csv")

# 念のため数値化（文字列混入対策）
mc = pd.to_numeric(df["material_concentration"], errors="coerce")
prop = pd.to_numeric(df["proportion"], errors="coerce") / 100.0  # % -> 比率
mw = pd.to_numeric(df["ExactMolWt"], errors="coerce")

mol = mc * prop / mw

insert_at = df.columns.get_loc("proportion") + 1
df.insert(insert_at, "mol", mol)

df_step1 = df

### step2:composition_idごとにtotal_molを計算

In [3]:
df_step2 = df_step1.copy()

total_mol = df_step2.groupby("composition_id")["mol"].transform("sum")

if "total_mol" in df_step2.columns:
    df_step2.drop(columns="total_mol", inplace=True)

df_step2.insert(df_step2.columns.get_loc("mol") + 1, "total_mol", total_mol)

### step3:各行ごとにmol_fractionを計算

In [5]:
df_step3 = df_step2.copy()

bad = df_step3["total_mol"].isna() | (df_step3["total_mol"] == 0)
if bad.any():
    raise ValueError(f"total_mol が 0 または NaN の行があります: {bad.sum()} 行")

mol_fraction = df_step3["mol"] / df_step3["total_mol"]

if "mol_fraction" in df_step3.columns:
    df_step3.drop(columns="mol_fraction", inplace=True)

df_step3.insert(df_step3.columns.get_loc("total_mol") + 1, "mol_fraction", mol_fraction)

### step4:各行ごとに記述子をmol_fractionで重み付けしてweighted_記述子列を作成

In [6]:
df_step4 = df_step3.copy()

# 重み付け対象（元のRDKit記述子列のみ：std_/range_/max_/min_ は除外）
exclude = {
    "id", "experiment_id", "composition_id", "material_id", "structure_id",
    "material_concentration", "proportion", "mol", "total_mol", "mol_fraction",
    "humidity", "temperature", "wvp",
}
base_desc_cols = df_step4.select_dtypes("number").columns.difference(exclude)

w = df_step4[base_desc_cols].fillna(0.0).mul(df_step4["mol_fraction"], axis=0)
w.columns = ["weighted_" + c for c in base_desc_cols]

# 再実行対策（既存 weighted_ を消してから追加）
df_step4 = df_step4.loc[:, ~df_step4.columns.str.startswith("weighted_")]
df_step4 = pd.concat([df_step4, w], axis=1)

### step5:composition_idごとに各記述子の分散（std）、範囲（range）、最大・最小値（max/min）を計算し、その分、列を追加

In [7]:
df_step5 = df_step4.copy()

# 再実行対策（前回作った集計列があれば消す）
df_step5 = df_step5.loc[:, ~df_step5.columns.str.startswith(("std_", "range_", "max_", "min_"))]

wcols = [c for c in df_step5.columns if c.startswith("weighted_")]

# 0.0 を無効扱いにする（計算用）
wdesc = df_step5[wcols].mask(df_step5[wcols].eq(0.0))

g = wdesc.groupby(df_step5["composition_id"])

mx = g.transform("max").fillna(0.0)
mn = g.transform("min").fillna(0.0)
rng = (mx - mn)                      # 0個/1個 -> 0.0 になる
std = g.transform("std").fillna(0.0) # 0個/1個 -> 0.0

mx.columns = ["max_" + c for c in wcols]
mn.columns = ["min_" + c for c in wcols]
rng.columns = ["range_" + c for c in wcols]
std.columns = ["std_" + c for c in wcols]

df_step5 = pd.concat([df_step5, std, rng, mx, mn], axis=1)

### step6:composition_idごとに足して１つの行にする

In [8]:
df_step6_src = df_step5.copy()

drop_cols = [
    "id", "material_id", "name", "structure_id", "material_concentration",
    "smiles", "proportion", "mol", "mol_fraction", "total_mol",
] + list(base_desc_cols)

df_step6_src = df_step6_src.drop(columns=[c for c in drop_cols if c in df_step6_src.columns])

weighted_cols = [c for c in df_step6_src.columns if c.startswith("weighted_")]
other_cols = [c for c in df_step6_src.columns if c not in (["composition_id"] + weighted_cols)]

df_step6 = (
    df_step6_src
    .groupby("composition_id", as_index=False)
    .agg({**{c: "first" for c in other_cols}, **{c: "sum" for c in weighted_cols}})
)

if "wvp" in df_step6.columns:
    wvp = df_step6.pop("wvp")
    df_step6["wvp"] = wvp

C:\Users\scarl\AppData\Local\Temp\ipykernel_45848\145860018.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .agg({**{c: "first" for c in other_cols}, **{c: "sum" for c in weighted_cols}})
C:\Users\scarl\AppData\Local\Temp\ipykernel_45848\145860018.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_step6["wvp"] = wvp


### step7:tocsv

In [9]:
out_path = r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\processed\260625\processed_data.csv"
df_step6.to_csv(out_path, index=False)
out_path

'C:\\Users\\scarl\\Downloads\\大学関連\\25講義関連\\b3後期\\b4\\coating_ML2\\coating_ML_260618\\data\\processed\\260625\\processed_data.csv'